# EC2 Code Checks Demonstration

This notebook demonstrates the three main code checks for reinforced concrete sections:

1. **BendingCheck** (EC2 §6.1) - Flexural capacity using M-N interaction diagram
2. **ShearCheck** (EC2 §6.2) - Variable strut inclination method for shear
3. **CrackingCheck** (EC2 §7.3) - Serviceability crack width calculation

All checks use first-principles strain compatibility and force equilibrium, implemented using fibre-based integration.

In [1]:
# Imports
import numpy as np
import plotly.graph_objects as go

# Section geometry
from materials.reinforced_concrete.geometry import (
    create_rectangular_section,
    create_linear_rebar_layer,
)

# Materials
from materials.reinforced_concrete.materials import (
    ConcreteMaterial,
    Rebar,
    ShearRebar,
)

# Code checks
from materials.reinforced_concrete.code_checks.ec2_2004 import (
    BendingCheck,
    ShearCheck,
    LoadCase,
    CrackingCheck,
    LoadDuration,
)

## 1. Create a Reinforced Concrete Section

We'll create a typical 300mm x 600mm rectangular beam section with:
- 3H25 tension reinforcement (bottom)
- 2H16 compression reinforcement (top)
- H10@200 shear links

In [2]:
# Section dimensions
width = 300  # mm
height = 600  # mm
cover = 35   # mm (to links, not the longitudinal bars)

# Create section
section = create_rectangular_section(
    width=width,
    height=height,
    section_name=f"{width}x{height} Beam",
)

# Define reinforcement
link_diameter = 10
tension_bar_diameter = 25
compression_bar_diameter = 16

# Calculate bar positions (y-coordinate from bottom)
y_tension = cover + link_diameter + tension_bar_diameter / 2
y_compression = height - cover - link_diameter - compression_bar_diameter / 2

# Side cover for main bars
side_cover = cover + link_diameter

# Tension reinforcement (at bottom) - using create_linear_rebar_layer
tension_rebar = Rebar(diameter=tension_bar_diameter)
num_btm_bars = 3
tension_layer = create_linear_rebar_layer(
    rebar=tension_rebar,
    n_bars=num_btm_bars,
    start_point=(side_cover + tension_bar_diameter / 2, y_tension),
    end_point=(width - side_cover - tension_bar_diameter / 2, y_tension),
)
section.add_rebar_group(tension_layer)

# Compression reinforcement (at top)
compression_rebar = Rebar(diameter=compression_bar_diameter)
num_top_bars = 2
compression_layer = create_linear_rebar_layer(
    rebar=compression_rebar,
    n_bars=num_top_bars,
    start_point=(side_cover + compression_bar_diameter / 2, y_compression),
    end_point=(width - side_cover - compression_bar_diameter / 2, y_compression),
)
section.add_rebar_group(compression_layer)

# Shear reinforcement (H10@200, 2-leg links)
shear_rebar = ShearRebar(
    diameter=link_diameter,
    link_spacing=200,
    n_legs=2,
)

# Define concrete material
concrete = ConcreteMaterial(grade="C30/37")

section.plot(concrete=concrete)

print(f"Section: {section.section_name}")
print(f"Dimensions: {width}mm x {height}mm")
print(f"Concrete: {concrete.grade} (f_ck = {concrete.f_ck} MPa, f_cd = {concrete.f_cd:.2f} MPa)")
print(f"Tension steel: {num_btm_bars}H{tension_bar_diameter} (A_s = {num_btm_bars * tension_rebar.area:.0f} mm²)")
print(f"Compression steel: {num_top_bars}H{compression_bar_diameter} (A_s' = {num_top_bars * compression_rebar.area:.0f} mm²)")
print(f"Shear links: H{link_diameter}@{shear_rebar.link_spacing} ({shear_rebar.n_legs}-leg)")
print(f"Reinforcement ratio: {section.reinforcement_ratio:.3%}")

Section: 300x600 Beam
Dimensions: 300mm x 600mm
Concrete: C30/37 (f_ck = 30.0 MPa, f_cd = 20.00 MPa)
Tension steel: 3H25 (A_s = 1473 mm²)
Compression steel: 2H16 (A_s' = 402 mm²)
Shear links: H10@200.0 (2-leg)
Reinforcement ratio: 1.042%


---
## 2. Bending Check (EC2 §6.1)

The `BendingCheck` uses an M-N interaction diagram to check flexural capacity.

### 2.1 Create the Check and Visualize the M-N Diagram

In [3]:
# Create bending check (diagram is built on initialization)
bending_check = BendingCheck(
    section=section,
    concrete=concrete,
)

# Plot M-N interaction diagram
bending_check.plot_mn(
    title=f"M-N Interaction Diagram - {width}x{height} Beam",
    show_metadata=True,
    show=False,
)

### 2.2 Check Multiple Load Cases

In [4]:
# Define load cases to check
load_cases = [
    {"name": "LC1: Pure bending", "M_Ed": 250, "N_Ed": 0},
    {"name": "LC2: +Bending + compression", "M_Ed": 200, "N_Ed": 1200},
    {"name": "LC3: -Bending + compression", "M_Ed": -150, "N_Ed": 1500},
    {"name": "LC4: +Bending + tension", "M_Ed": 150, "N_Ed": -400},
    {"name": "LC5: High utilization", "M_Ed": 390, "N_Ed": 600},
    {"name": "LC6: Failure", "M_Ed": -200, "N_Ed": -500},
]

print("Bending Check Results (EC2 §6.1)")
print("=" * 70)

results = []
for lc in load_cases:
    result = bending_check.perform_check(M_Ed=lc["M_Ed"], N_Ed=lc["N_Ed"])
    results.append({**lc, "result": result})
    
    status = result.status.value.upper()
    print(f"\n{lc['name']}:")
    print(f"  M_Ed = {lc['M_Ed']:6.1f} kN·m, N_Ed = {lc['N_Ed']:6.1f} kN")
    print(f"  M_Rd = {result.details['M_Rd']:6.1f} kN·m, N_Rd = {result.details['N_Rd']:6.1f} kN")
    print(f"  Utilization: {result.utilization:.1%} [{status}]")

Bending Check Results (EC2 §6.1)

LC1: Pure bending:
  M_Ed =  250.0 kN·m, N_Ed =    0.0 kN
  M_Rd =  324.7 kN·m, N_Rd =    0.0 kN
  Utilization: 77.0% [PASS]

LC2: +Bending + compression:
  M_Ed =  200.0 kN·m, N_Ed = 1200.0 kN
  M_Rd =  347.0 kN·m, N_Rd = 2081.8 kN
  Utilization: 57.6% [PASS]

LC3: -Bending + compression:
  M_Ed = -150.0 kN·m, N_Ed = 1500.0 kN
  M_Rd = -332.1 kN·m, N_Rd = 3320.5 kN
  Utilization: 45.2% [PASS]

LC4: +Bending + tension:
  M_Ed =  150.0 kN·m, N_Ed = -400.0 kN
  M_Rd =  206.7 kN·m, N_Rd = -551.2 kN
  Utilization: 72.6% [PASS]

LC5: High utilization:
  M_Ed =  390.0 kN·m, N_Ed =  600.0 kN
  M_Rd =  426.8 kN·m, N_Rd =  656.6 kN
  Utilization: 91.4% [PASS]

LC6: Failure:
  M_Ed = -200.0 kN·m, N_Ed = -500.0 kN
  M_Rd =  -62.1 kN·m, N_Rd = -155.1 kN
  Utilization: 322.3% [FAIL]


### 2.3 Visualize Load Cases on M-N Diagram

In [5]:
# Plot diagram with load points
plot_points = [
    {"N_Ed": lc["N_Ed"], "M_Ed": lc["M_Ed"], "name": lc["name"]}
    for lc in load_cases
]

bending_check.plot_mn(
    load_points=plot_points,
    show_vectors=True,
    title="M-N Diagram with Load Cases",
    show=False,
    width=1000,
)

### 2.4 Stress-Strain Visualization

Visualize the strain profile and stress distribution for a specific load case.

In [6]:
# Visualize stress-strain for LC1 (pure bending)
bending_check.plot_stress_strain(
    M_Ed=250,
    N_Ed=100,
    title="Stress-Strain Distribution (M=250 kN·m, N=0)",
    section_render="filled",
    height=800,
    show=False,
)

### 2.5 Get Moment Capacity at a Given Axial Force

In [7]:
# Get moment capacity for different axial loads
axial_loads = [0, 200, 500, 1000, -100, -300]

print("Moment Capacity at Various Axial Loads")
print("=" * 50)
print(f"{'N_Ed (kN)':>12} | {'M_Rd+ (kN·m)':>14} | {'M_Rd- (kN·m)':>14}")
print("-" * 50)

for N_Ed in axial_loads:
    M_pos, M_neg = bending_check.get_moment_capacity(N_Ed=N_Ed)
    M_pos_str = f"{M_pos:.1f}" if M_pos is not None else "N/A"
    M_neg_str = f"{M_neg:.1f}" if M_neg is not None else "N/A"
    print(f"{N_Ed:>12.0f} | {M_pos_str:>14} | {M_neg_str:>14}")

Moment Capacity at Various Axial Loads
   N_Ed (kN) |   M_Rd+ (kN·m) |   M_Rd- (kN·m)
--------------------------------------------------
           0 |          324.8 |          -99.4
         200 |          363.4 |         -147.5
         500 |          408.9 |         -218.8
        1000 |          452.7 |         -331.6
        -100 |          303.9 |          -75.3
        -300 |          261.1 |          -27.2


---
## 3. Shear Check (EC2 §6.2)

The `ShearCheck` implements the Variable Strut Inclination Method for shear design.

### 3.1 Create the Check

In [8]:
# Create shear check with rigorous mode (accurate lever arm from force centroids)
shear_check = ShearCheck(
    section=section,
    concrete=concrete,
    shear_reinforcement=shear_rebar,
    use_mechanical_lever_arm=True,  # Use accurate lever arm calculation
)

print(f"Shear Check Configuration")
print(f"=" * 40)
print(f"Web breadth b_w: {shear_check.breadth:.0f} mm")
print(f"Design concrete strength f_cd: {shear_check.f_cd_design:.2f} MPa")
print(f"Design yield strength f_ywd: {shear_check.f_ywd_design:.0f} MPa")
print(f"Mode: {'Rigorous' if shear_check.use_mechanical_lever_arm else 'Approximate'} (z = {'computed' if shear_check.use_mechanical_lever_arm else '0.9d'})")

Shear Check Configuration
Web breadth b_w: 300 mm
Design concrete strength f_cd: 20.00 MPa
Design yield strength f_ywd: 435 MPa
Mode: Rigorous (z = computed)


### 3.2 Check Multiple Shear Load Cases

In [9]:
# Define shear load cases
shear_load_cases = [
    {"name": "Support - high shear", "V_Ed": 250, "M_Ed": 50, "N_Ed": 0},
    {"name": "Midspan - low shear", "V_Ed": 80, "M_Ed": 300, "N_Ed": 0},
    {"name": "Column face", "V_Ed": 300, "M_Ed": 100, "N_Ed": 500},
    {"name": "Tension member", "V_Ed": 150, "M_Ed": 100, "N_Ed": -200},
]

print("Shear Check Results (EC2 §6.2)")
print("=" * 90)

for lc in shear_load_cases:
    load_case = LoadCase(V_Ed=lc["V_Ed"], M_Ed=lc["M_Ed"], N_Ed=lc["N_Ed"])
    result = shear_check.perform_check(load_case=load_case)
    
    status = result.status.value.upper()
    d = result.details
    
    print(f"\n{lc['name']}:")
    print(f"  V_Ed = {lc['V_Ed']:5.0f} kN, M_Ed = {lc['M_Ed']:5.0f} kN·m, N_Ed = {lc['N_Ed']:5.0f} kN")
    print(f"  V_Rd,c = {d['V_Rd_c']:5.1f} kN (concrete)")
    print(f"  V_Rd,s = {d['V_Rd_s']:5.1f} kN (reinforcement)")
    print(f"  V_Rd,max = {d['V_Rd_max']:5.1f} kN (strut crushing)")
    print(f"  Governing: {d['governing_mode']}")
    print(f"  θ = {d['theta_deg']:.1f}° (cot θ = {d['cot_theta']:.2f})")
    print(f"  Utilization: {result.utilization:.1%} [{status}]")

Shear Check Results (EC2 §6.2)

Support - high shear:
  V_Ed =   250 kN, M_Ed =    50 kN·m, N_Ed =     0 kN
  V_Rd,c =  94.3 kN (concrete)
  V_Rd,s = 411.2 kN (reinforcement)
  V_Rd,max = 763.0 kN (strut crushing)
  Governing: shear reinforcement (V_Rd,s)
  θ = 21.8° (cot θ = 2.50)
  Utilization: 60.8% [PASS]

Midspan - low shear:
  V_Ed =    80 kN, M_Ed =   300 kN·m, N_Ed =     0 kN
  V_Rd,c =  94.3 kN (concrete)
  V_Rd,s = 405.2 kN (reinforcement)
  V_Rd,max = 751.9 kN (strut crushing)
  Governing: concrete shear (V_Rd,c,cracked)
  θ = 21.8° (cot θ = 2.50)
  Utilization: 84.8% [PASS]

Column face:
  V_Ed =   300 kN, M_Ed =   100 kN·m, N_Ed =   500 kN
  V_Rd,c = 158.7 kN (concrete)
  V_Rd,s = 347.9 kN (reinforcement)
  V_Rd,max = 645.6 kN (strut crushing)
  Governing: shear reinforcement (V_Rd,s)
  θ = 21.8° (cot θ = 2.50)
  Utilization: 86.2% [PASS]

Tension member:
  V_Ed =   150 kN, M_Ed =   100 kN·m, N_Ed =  -200 kN
  V_Rd,c =  68.6 kN (concrete)
  V_Rd,s = 416.8 kN (reinforcement

C:\Users\user\Repo\Scripts\section_design_checks\materials\reinforced_concrete\code_checks\ec2_2004\shear_check.py:495: UserWarning:

Lever arm capped: z_mech=501.6mm > 0.9d=488.2mm. Using z=0.9d for EC2 truss model.



### 3.3 Compare Rigorous vs Approximate Mode

In [10]:
# Create approximate mode check for comparison
shear_check_approx = ShearCheck(
    section=section,
    concrete=concrete,
    shear_reinforcement=shear_rebar,
    use_mechanical_lever_arm=False,  # Always uses z = 0.9d
)

# Compare for a single load case
test_case = LoadCase(V_Ed=250, M_Ed=200, N_Ed=100)

result_rigorous = shear_check.perform_check(load_case=test_case)
result_approx = shear_check_approx.perform_check(load_case=test_case)

print("Comparison: Rigorous vs Approximate Mode")
print("=" * 50)
print(f"Load: V_Ed={test_case.V_Ed} kN, M_Ed={test_case.M_Ed} kN·m, N_Ed={test_case.N_Ed} kN")
print()
print(f"{'Parameter':<20} | {'Rigorous':>12} | {'Approximate':>12}")
print("-" * 50)
print(f"{'Lever arm z (mm)':<20} | {result_rigorous.details['z']:>12.1f} | {result_approx.details['z']:>12.1f}")
print(f"{'V_Rd,s (kN)':<20} | {result_rigorous.details['V_Rd_s']:>12.1f} | {result_approx.details['V_Rd_s']:>12.1f}")
print(f"{'V_Rd,max (kN)':<20} | {result_rigorous.details['V_Rd_max']:>12.1f} | {result_approx.details['V_Rd_max']:>12.1f}")
print(f"{'Utilization':<20} |  {result_rigorous.utilization:>11.1%} |  {result_approx.utilization:>11.1%}")

Comparison: Rigorous vs Approximate Mode
Load: V_Ed=250.0 kN, M_Ed=200.0 kN·m, N_Ed=100.0 kN

Parameter            |     Rigorous |  Approximate
--------------------------------------------------
Lever arm z (mm)     |        472.0 |        488.2
V_Rd,s (kN)          |        403.0 |        416.8
V_Rd,max (kN)        |        747.7 |        773.4
Utilization          |        62.0% |        60.0%


### 3.4 Design Helper: Required Shear Reinforcement

In [14]:
# Calculate required shear reinforcement for different shear forces
shear_forces = [100, 150, 200, 250, 300, 350]

print("Required Shear Reinforcement (A_sw/s)")
print("=" * 55)
print(f"{'V_Ed (kN)':>10} | {'A_sw/s (mm²/mm)':>18} | {'Equiv. spacing (mm)':>20}")
print("-" * 55)

A_sw_link = shear_rebar.n_legs * np.pi * (shear_rebar.diameter ** 2) / 4

for V_Ed in shear_forces:
    A_sw_over_s = shear_check.get_required_shear_reinforcement(
        V_Ed=V_Ed, M_Ed=100, N_Ed=0, cot_theta=2.5
    )
    equiv_spacing = A_sw_link / A_sw_over_s if A_sw_over_s > 0 else float('inf')
    
    if A_sw_over_s == 0:
        print(f"{V_Ed:>10.0f} | {'Not required':>18} | {'N/A':>20}")
    else:
        print(f"{V_Ed:>10.0f} | {A_sw_over_s:>18.4f} | {equiv_spacing:>20.0f}")

Required Shear Reinforcement (A_sw/s)
 V_Ed (kN) |    A_sw/s (mm²/mm) |  Equiv. spacing (mm)
-------------------------------------------------------
       100 |             0.2629 |                  597
       150 |             0.2871 |                  547
       200 |             0.3828 |                  410
       250 |             0.4785 |                  328
       300 |             0.5742 |                  274
       350 |             0.6700 |                  234


### 3.5 Section Without Shear Reinforcement

In [26]:
# Check capacity without shear reinforcement
shear_check_unreinforced = ShearCheck(
    section=section,
    concrete=concrete,
    shear_reinforcement=None,  # No shear links
)

test_cases = [
    LoadCase(V_Ed=50, M_Ed=100, N_Ed=0),
    LoadCase(V_Ed=100, M_Ed=100, N_Ed=0),
    LoadCase(V_Ed=80, M_Ed=50, N_Ed=300),  # Compression helps V_Rd,c
]

print("Unreinforced Shear Capacity (V_Rd,c only)")
print("=" * 72)

for tc in test_cases:
    result = shear_check_unreinforced.perform_check(load_case=tc)
    status = result.status.value.upper()
    print(f"V_Ed = {tc.V_Ed:3.0f} kN, N_Ed = {tc.N_Ed:3.0f} kN:   "
          f"V_Rd,c = {result.details['V_Rd_c']:5.1f} kN, "
          f"Util = {result.utilization:6.1%} [{status}]")

Unreinforced Shear Capacity (V_Rd,c only)
V_Ed =  50 kN, N_Ed =   0 kN:   V_Rd,c =  94.3 kN, Util =  53.0% [PASS]
V_Ed = 100 kN, N_Ed =   0 kN:   V_Rd,c =  94.3 kN, Util = 106.0% [FAIL]
V_Ed =  80 kN, N_Ed = 300 kN:   V_Rd,c = 133.0 kN, Util =  60.2% [PASS]


---
## 4. Cracking Check (EC2 §7.3)

The `CrackingCheck` calculates crack widths for serviceability limit state.

### 4.1 Create the Check

In [32]:
# Create cracking check
cracking_check = CrackingCheck(
    section=section,
    concrete=concrete,
    w_k_limit=0.3,  # mm (typical for XC2/XC3 exposure)
    load_duration=LoadDuration.LONG_TERM,  # k_t = 0.4
)

print("Cracking Check Configuration")
print("=" * 40)
print(f"Crack width limit w_k,lim: {cracking_check.w_k_limit} mm")
print(f"Load duration: {cracking_check.load_duration.value} (k_t = {cracking_check.k_t})")
print(f"Bond coefficient k_1: {cracking_check.find_k_1()}")

Cracking Check Configuration
Crack width limit w_k,lim: 0.3 mm
Load duration: long_term (k_t = 0.4)
Bond coefficient k_1: 0.8


### 4.2 Check Crack Widths for Multiple Load Cases

In [33]:
# Define SLS load cases (characteristic or quasi-permanent)
sls_load_cases = [
    {"name": "Low moment", "M_Ed": 80, "N_Ed": 0},
    {"name": "Service moment", "M_Ed": 150, "N_Ed": 0},
    {"name": "High moment", "M_Ed": 200, "N_Ed": 0},
    {"name": "With compression", "M_Ed": 150, "N_Ed": 200},
    {"name": "With tension", "M_Ed": 100, "N_Ed": -100},
]

M_cr = cracking_check.find_cracking_moment()

print("Cracking Check Results (EC2 §7.3)")
print("=" * 80)

for lc in sls_load_cases:
    result = cracking_check.perform_check(M_Ed=lc["M_Ed"], N_Ed=lc["N_Ed"])
    d = result.details
    
    status = result.status.value.upper()
    
    print(f"\n{lc['name']}:")
    print(f"  M_Ed = {lc['M_Ed']:.0f} kN·m, N_Ed = {lc['N_Ed']:.0f} kN")
    
    if d.get('is_cracked', True):
        print(f"  Section is CRACKED")
        print(f"  σ_s = {d.get('sigma_s', 0):.1f} MPa (steel stress)")
        print(f"  s_r,max = {d.get('s_r_max', 0):.1f} mm (crack spacing)")
        print(f"  w_k = {d.get('w_k', 0):.3f} mm ≤ {cracking_check.w_k_limit} mm")
        print(f"  Utilization: {result.utilization:.1%} [{status}]")
    else:
        print(f"  Section is UNCRACKED (M_Ed < M_cr)")
        print(f"  w_k = 0 mm [{status}]")

Cracking Check Results (EC2 §7.3)

Low moment:
  M_Ed = 80 kN·m, N_Ed = 0 kN
  Section is CRACKED
  σ_s = 114.5 MPa (steel stress)
  s_r,max = 265.7 mm (crack spacing)
  w_k = 0.102 mm ≤ 0.3 mm
  Utilization: 34.0% [PASS]

Service moment:
  M_Ed = 150 kN·m, N_Ed = 0 kN
  Section is CRACKED
  σ_s = 214.6 MPa (steel stress)
  s_r,max = 265.7 mm (crack spacing)
  w_k = 0.235 mm ≤ 0.3 mm
  Utilization: 78.3% [PASS]

High moment:
  M_Ed = 200 kN·m, N_Ed = 0 kN
  Section is CRACKED
  σ_s = 286.1 MPa (steel stress)
  s_r,max = 265.7 mm (crack spacing)
  w_k = 0.330 mm ≤ 0.3 mm
  Utilization: 110.0% [FAIL]

With compression:
  M_Ed = 150 kN·m, N_Ed = 200 kN
  Section is CRACKED
  σ_s = 158.4 MPa (steel stress)
  s_r,max = 250.1 mm (crack spacing)
  w_k = 0.156 mm ≤ 0.3 mm
  Utilization: 52.1% [PASS]

With tension:
  M_Ed = 100 kN·m, N_Ed = -100 kN
  Section is CRACKED
  σ_s = 173.8 MPa (steel stress)
  s_r,max = 276.7 mm (crack spacing)
  w_k = 0.184 mm ≤ 0.3 mm
  Utilization: 61.3% [PASS]


### 4.3 Detailed Cracking Results

In [35]:
# Get detailed cracking results for a specific case
detailed = cracking_check.calculate_detailed(M_Ed=150, N_Ed=0)

print("Detailed Cracking Calculation (M_Ed = 150 kN·m)")
print("=" * 50)
print(f"Is cracked: {detailed.is_cracked}")
print(f"Neutral axis depth x: {detailed.x:.1f} mm" if detailed.x else "N/A")
print(f"Effective height h_c,ef: {detailed.h_c_ef:.1f} mm")
print(f"Tension steel area A_s: {detailed.rho_p_eff * detailed.h_c_ef * cracking_check.breadth:.0f} mm²")
print(f"Effective reinf. ratio ρ_p,eff: {detailed.rho_p_eff:.4f}")
print(f"Equivalent bar diameter φ_eq: {detailed.phi_eq:.1f} mm")
print(f"Cover to tension steel: {detailed.cover:.1f} mm")
print(f"Steel stress σ_s: {detailed.sigma_s:.1f} MPa")
print(f"Maximum crack spacing s_r,max: {detailed.s_r_max:.1f} mm")
print(f"Mean strain difference (ε_sm - ε_cm): {detailed.eps_sm_minus_eps_cm:.6f}")
print(f"Crack width w_k: {detailed.w_k:.3f} mm")
print(f"Limit w_k,lim: {detailed.w_k_limit} mm")
print(f"Utilization: {detailed.w_k / detailed.w_k_limit * 100:.1f} %")

Detailed Cracking Calculation (M_Ed = 150 kN·m)
Is cracked: True
Neutral axis depth x: 209.5 mm
Effective height h_c,ef: 130.2 mm
Tension steel area A_s: 1473 mm²
Effective reinf. ratio ρ_p,eff: 0.0377
Equivalent bar diameter φ_eq: 25.0 mm
Cover to tension steel: 45.0 mm
Steel stress σ_s: 214.6 MPa
Maximum crack spacing s_r,max: 265.7 mm
Mean strain difference (ε_sm - ε_cm): 0.000884
Crack width w_k: 0.235 mm
Limit w_k,lim: 0.3 mm
Utilization: 78.3 %


### 4.4 Compare Short-Term vs Long-Term Loading

In [43]:
# Short-term loading (k_t = 0.6)
cracking_check_short = CrackingCheck(
    section=section,
    concrete=concrete,
    w_k_limit=0.3,
    load_duration=LoadDuration.SHORT_TERM,
)

M_test = 150  # kN·m

result_long = cracking_check.perform_check(M_Ed=M_test, N_Ed=0)
result_short = cracking_check_short.perform_check(M_Ed=M_test, N_Ed=0)

print("Effect of Load Duration on Crack Width")
print(f"M_Ed = {M_test} kN·m, N_Ed = 0")
print("=" * 52)
print(f"{'Parameter':<25} | {'Long-term':>10} | {'Short-term':>10}")
print("-" * 52)
print(f"{'k_t factor':<25} | {cracking_check.k_t:>10.1f} | {cracking_check_short.k_t:>10.1f}")
print(f"{'w_k (mm)':<25} | {result_long.details['w_k']:>10.3f} | {result_short.details['w_k']:>10.3f}")
print(f"{'Utilization':<25} |   {result_long.utilization:>7.1%}  | {result_short.utilization:>10.1%}")

Effect of Load Duration on Crack Width
M_Ed = 150 kN·m, N_Ed = 0
Parameter                 |  Long-term | Short-term
----------------------------------------------------
k_t factor                |        0.4 |        0.6
w_k (mm)                  |      0.235 |      0.210
Utilization               |     78.3%  |      69.9%


### 4.5 Minimum Crack Control Reinforcement

In [44]:
# Calculate minimum reinforcement for crack control
A_s_min = cracking_check.find_minimum_crack_reinforcement(
    steel_stress=300,  # Typical SLS stress limit
    N_Ed=0,
    is_in_bending=True,
)

# Compare with provided reinforcement
A_s_provided = 3 * tension_rebar.area

print("Minimum Reinforcement for Crack Control (EC2 §7.3.2)")
print("=" * 50)
print(f"A_s,min required: {A_s_min:.0f} mm²")
print(f"A_s provided: {A_s_provided:.0f} mm²")
print(f"Ratio: {A_s_provided / A_s_min:.2f}x minimum")

Minimum Reinforcement for Crack Control (EC2 §7.3.2)
A_s,min required: 348 mm²
A_s provided: 1473 mm²
Ratio: 4.24x minimum


---
## 5. Combined Analysis: Full Design Check

Check all three criteria for a typical beam load case.

In [45]:
# Define ULS and SLS loads for a typical beam
# ULS (factored)
M_Ed_uls = 280  # kN·m
V_Ed_uls = 220  # kN
N_Ed_uls = 50   # kN (small compression)

# SLS (characteristic/quasi-permanent)
M_Ed_sls = 180  # kN·m
N_Ed_sls = 35   # kN

print("FULL DESIGN CHECK - 300x600 Beam")
print("=" * 60)
print(f"\nULS Loads: M_Ed = {M_Ed_uls} kN·m, V_Ed = {V_Ed_uls} kN, N_Ed = {N_Ed_uls} kN")
print(f"SLS Loads: M_Ed = {M_Ed_sls} kN·m, N_Ed = {N_Ed_sls} kN")

# Bending check
bending_result = bending_check.perform_check(M_Ed=M_Ed_uls, N_Ed=N_Ed_uls)
print(f"\n1. BENDING (EC2 §6.1)")
print(f"   M_Rd = {bending_result.details['M_Rd']:.1f} kN·m")
print(f"   Utilization: {bending_result.utilization:.1%} [{bending_result.status.value.upper()}]")

# Shear check
shear_result = shear_check.perform_check(
    load_case=LoadCase(V_Ed=V_Ed_uls, M_Ed=M_Ed_uls, N_Ed=N_Ed_uls)
)
print(f"\n2. SHEAR (EC2 §6.2)")
print(f"   V_Rd = {shear_result.details['V_Rd']:.1f} kN ({shear_result.details['governing_mode']})")
print(f"   Utilization: {shear_result.utilization:.1%} [{shear_result.status.value.upper()}]")

# Cracking check
cracking_result = cracking_check.perform_check(M_Ed=M_Ed_sls, N_Ed=N_Ed_sls)
print(f"\n3. CRACKING (EC2 §7.3)")
print(f"   w_k = {cracking_result.details.get('w_k', 0):.3f} mm ≤ {cracking_check.w_k_limit} mm")
print(f"   Utilization: {cracking_result.utilization:.1%} [{cracking_result.status.value.upper()}]")

# Summary
all_pass = all(r.status == "pass" for r in [bending_result, shear_result, cracking_result])
print(f"\n{'=' * 60}")
print(f"OVERALL: {'ALL CHECKS PASS' if all_pass else 'SOME CHECKS FAIL'}")

FULL DESIGN CHECK - 300x600 Beam

ULS Loads: M_Ed = 280 kN·m, V_Ed = 220 kN, N_Ed = 50 kN
SLS Loads: M_Ed = 180 kN·m, N_Ed = 35 kN

1. BENDING (EC2 §6.1)
   M_Rd = 336.8 kN·m
   Utilization: 83.1% [PASS]

2. SHEAR (EC2 §6.2)
   V_Rd = 403.9 kN (shear reinforcement (V_Rd,s))
   Utilization: 54.5% [PASS]

3. CRACKING (EC2 §7.3)
   w_k = 0.277 mm ≤ 0.3 mm
   Utilization: 92.3% [PASS]

OVERALL: ALL CHECKS PASS


---
## 6. Parametric Study: Crack Width vs Moment

In [49]:
# Generate crack width curve
M_cr = cracking_check.find_cracking_moment()
moments = np.linspace(120, 160, 5)
crack_widths = []

for M in moments:
    if M <= M_cr:
        crack_widths.append(0)
    else:
        try:
            result = cracking_check.calculate_detailed(M_Ed=float(M), N_Ed=0)
            crack_widths.append(result.w_k)
        except:
            crack_widths.append(np.nan)

# Plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=moments,
    y=crack_widths,
    mode='lines',
    name='Crack width w_k',
    line=dict(color='blue', width=2),
))

# Add limit line
fig.add_hline(
    y=cracking_check.w_k_limit,
    line_dash="dash",
    line_color="red",
    annotation_text=f"w_k,lim = {cracking_check.w_k_limit} mm",
)

# Add cracking moment line
fig.add_vline(
    x=M_cr,
    line_dash="dot",
    line_color="gray",
    annotation_text=f"M_cr = {M_cr:.0f} kN·m",
)

fig.update_layout(
    title="Crack Width vs Applied Moment",
    xaxis_title="Moment M_Ed (kN·m)",
    yaxis_title="Crack Width w_k (mm)",
    height=500,
)

fig.show()

---
## 7. Features of SLS Linear Elastic Constitutive Model

The `CrackingCheck` uses a **Linear Elastic** concrete stress-strain model by default, which is appropriate for Serviceability Limit State (SLS) analysis where concrete stresses remain in the elastic range.

### Key Features:
- `ConcreteModelType.LINEAR_ELASTIC` - Linear σ = E × ε relationship
- Configurable elastic modulus (default: `E_cm`, or `E_cm_eff` with creep)
- Optional tension capacity up to `f_ctm` (brittle cutoff)

In [52]:
# Import the ConcreteModelType to show it's LINEAR_ELASTIC by default
from materials.reinforced_concrete.constitutive import ConcreteModelType

# CrackingCheck uses LINEAR_ELASTIC by default for SLS
print("Cracking Check Constitutive Model Configuration")
print("=" * 50)
print(f"Default concrete model: {cracking_check.concrete_model_type}")
print(f"Model type enum: {ConcreteModelType.LINEAR_ELASTIC}")

# The effective modulus includes creep reduction
print(f"\nEffective Modulus (with φ={cracking_check.creep_coefficient}):")
print(f"  E_cm (short-term): {concrete.E_cm:.0f} MPa")
print(f"  E_c,eff (long-term): {cracking_check.E_c_eff:.0f} MPa")
print(f"  Reduction factor: 1/(1+φ) = {1/cracking_check.effective_modulus_ratio:.3f}")

# Compare with ULS checks that CANNOT use LINEAR_ELASTIC
print("\n⚠️ Note: LINEAR_ELASTIC is only valid for SLS checks.")
print("   ULS checks (BendingCheck, ShearCheck) must use:")
print(f"   - {ConcreteModelType.PARABOLA_RECTANGLE} (default)")
print(f"   - {ConcreteModelType.BILINEAR}")


Cracking Check Constitutive Model Configuration
Default concrete model: linear-elastic
Model type enum: linear-elastic

Effective Modulus (with φ=1.5):
  E_cm (short-term): 32837 MPa
  E_c,eff (long-term): 13135 MPa
  Reduction factor: 1/(1+φ) = 0.400

⚠️ Note: LINEAR_ELASTIC is only valid for SLS checks.
   ULS checks (BendingCheck, ShearCheck) must use:
   - parabola-rectangle (default)
   - bilinear


---
## 8. Stress Limitation Checks (EC2 §7.2)

The `CrackingCheck` includes automatic stress limitation checks per EC2 §7.2:

| Check | Limit | Warning Condition | Consequence |
|-------|-------|-------------------|-------------|
| §7.2(2) | k₁·f_ck | σ_c > limit under **characteristic** loads | Longitudinal cracking risk |
| §7.2(3) | k₂·f_ck | σ_c > limit under **quasi-permanent** loads | Non-linear creep threshold |
| §7.2(4)P | f_yk | σ_s > limit | Inelastic strain |
| §7.2(5) | k₃·f_yk | σ_s > limit under **characteristic** loads | Possible unacceptable appearance |

Default NDP values: k₁ = 0.6, k₂ = 0.45, k₃ = 0.8

In [53]:
import warnings
from materials.reinforced_concrete.ndp import get_ndp

# Create cracking checks with different stress limit flags
cracking_char = CrackingCheck(
    section=section,
    concrete=concrete,
    w_k_limit=0.3,
    load_duration=LoadDuration.LONG_TERM,
    check_k1_stress=True,  # Enable characteristic stress limit
)

cracking_qp = CrackingCheck(
    section=section,
    concrete=concrete,
    w_k_limit=0.3,
    load_duration=LoadDuration.LONG_TERM,
    check_k2_stress=True,  # Enable quasi-permanent stress limit (default)
)

# NDP stress factors
k_1 = get_ndp("k_1_stress")
k_2 = get_ndp("k_2_stress")
k_3 = get_ndp("k_3_stress")

print("Stress Limitation Parameters (EC2 §7.2)")
print("=" * 60)
print(f"Concrete: {concrete.grade} (f_ck = {concrete.f_ck} MPa)")
print(f"\nCharacteristic stress limit (k_1·f_ck):")
print(f"  k_1 = {k_1}, limit = {k_1 * concrete.f_ck:.1f} MPa")
print(f"\nQuasi-permanent stress limit (k_2·f_ck):")
print(f"  k_2 = {k_2}, limit = {k_2 * concrete.f_ck:.1f} MPa")
print(f"\nReinforcement stress limit (k_3·f_yk):")
f_yk = 500  # Typical B500B
print(f"  k_3 = {k_3}, limit = {k_3 * f_yk:.0f} MPa")

# Demonstrate a case that triggers warnings
print("\n" + "=" * 60)
print("Testing stress limit warnings with high moment (M=220 kN·m):")
print("-" * 60)

# Capture warnings to display them
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    result_high = cracking_qp.perform_check(M_Ed=220, N_Ed=0)
    
    if w:
        print("Warnings triggered:")
        for warning in w:
            print(f"   {warning.message}")
    else:
        print("No stress limit warnings triggered")

print(f"\nResult details:")
print(f"  sigma_c,peak = {result_high.details.get('sigma_c_peak', 0):.1f} MPa")
print(f"  k_2*f_ck limit = {k_2 * concrete.f_ck:.1f} MPa")
print(f"  sigma_s = {result_high.details.get('sigma_s', 0):.1f} MPa")
print(f"  w_k = {result_high.details.get('w_k', 0):.3f} mm")

Stress Limitation Parameters (EC2 §7.2)
Concrete: C30/37 (f_ck = 30.0 MPa)

Characteristic stress limit (k_1·f_ck):
  k_1 = 0.6, limit = 18.0 MPa

Quasi-permanent stress limit (k_2·f_ck):
  k_2 = 0.45, limit = 13.5 MPa

Reinforcement stress limit (k_3·f_yk):
  k_3 = 0.8, limit = 400 MPa

Testing stress limit warnings with high moment (M=220 kN·m):
------------------------------------------------------------
No stress limit warnings triggered

Result details:
  sigma_c,peak = 12.4 MPa
  k_2*f_ck limit = 13.5 MPa
  sigma_s = 314.7 MPa
  w_k = 0.368 mm


---
## 9. Non-Linear Creep Adjustment (EC2 §3.1.4(4))

When concrete stress exceeds the quasi-permanent limit (σ_c > k₂·f_ck), non-linear creep effects become significant. The `CrackingCheck` can automatically adjust the creep coefficient:

**Formula (EC2 Eq. 3.7):**
$$\varphi_{NL} = \varphi \cdot \exp(1.5 \cdot (k_\sigma - 0.45))$$

where k_σ = σ_c / f_cm

This affects the effective modulus: E_cm,eff = E_cm / (1 + φ_NL)

### Options:
- `apply_nonlinear_creep=True` (default): Auto-adjust when stress exceeds limit
- `iterate_nonlinear_creep=True`: Iterate until convergence (max 5 iterations)

In [56]:
# Compare linear vs non-linear creep for high stress case
M_test_high = 280  # kN·m - high enough to exceed k_2·f_ck limit

# Without non-linear creep adjustment
cracking_linear = CrackingCheck(
    section=section,
    concrete=concrete,
    w_k_limit=0.3,
    load_duration=LoadDuration.LONG_TERM,
    creep_coefficient=2.0,  # Higher creep coefficient for demonstration
    apply_nonlinear_creep=False,  # Disable non-linear creep
)

# With non-linear creep adjustment (single iteration)
cracking_nonlinear = CrackingCheck(
    section=section,
    concrete=concrete,
    w_k_limit=0.3,
    load_duration=LoadDuration.LONG_TERM,
    creep_coefficient=2.0,
    apply_nonlinear_creep=True,  # Enable non-linear creep (default)
    iterate_nonlinear_creep=False,  # Single adjustment
)

# With iterated non-linear creep
cracking_iterated = CrackingCheck(
    section=section,
    concrete=concrete,
    w_k_limit=0.3,
    load_duration=LoadDuration.LONG_TERM,
    creep_coefficient=2.0,
    apply_nonlinear_creep=True,
    iterate_nonlinear_creep=True,  # Iterate to convergence
)

# Suppress warnings for cleaner output
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    result_linear = cracking_linear.calculate_detailed(M_Ed=M_test_high, N_Ed=0)
    result_nonlinear = cracking_nonlinear.calculate_detailed(M_Ed=M_test_high, N_Ed=0)
    result_iterated = cracking_iterated.calculate_detailed(M_Ed=M_test_high, N_Ed=0)

print(f"Non-Linear Creep Comparison (M_Ed = {M_test_high} kN·m)")
print("=" * 79)
print(f"Quasi-permanent stress limit: k_2·f_ck = {k_2 * concrete.f_ck:.1f} MPa")
print()
print(f"{'Parameter':<35} | {'Linear phi':>10} | {'phi_NL (1x)':>10} | {'phi_NL (iter)':>10}")
print("-" * 79)
print(f"{'Creep coefficient used':<35} | {cracking_linear.creep_coefficient:>10.2f} | {result_nonlinear.creep_coefficient_used:>10.2f} | {result_iterated.creep_coefficient_used:>10.2f}")
print(f"{'Peak concrete stress (MPa)':<35} | {result_linear.sigma_c_peak:>10.1f} | {result_nonlinear.sigma_c_peak:>10.1f} | {result_iterated.sigma_c_peak:>10.1f}")
print(f"{'Non-linear creep applied?':<35} | {'No':>10} | {'Yes':>10} | {'Yes':>10}")
print(f"{'Steel stress (MPa)':<35} | {result_linear.sigma_s:>10.1f} | {result_nonlinear.sigma_s:>10.1f} | {result_iterated.sigma_s:>10.1f}")
print(f"{'Crack width w_k (mm)':<35} | {result_linear.w_k:>10.3f} | {result_nonlinear.w_k:>10.3f} | {result_iterated.w_k:>10.3f}")

# Show the effect on effective modulus
E_cm = concrete.E_cm
phi_linear = cracking_linear.creep_coefficient
phi_NL_single = result_nonlinear.creep_coefficient_used
phi_NL_iter = result_iterated.creep_coefficient_used

print()
print("Effective Modulus Comparison:")
print(f"  E_cm,eff (linear):    {E_cm / (1 + phi_linear):>8.0f} MPa (phi = {phi_linear:.2f})")
print(f"  E_cm,eff (phi_NL 1x):  {E_cm / (1 + phi_NL_single):>8.0f} MPa (phi_NL = {phi_NL_single:.2f})")
print(f"  E_cm,eff (phi_NL iter):{E_cm / (1 + phi_NL_iter):>8.0f} MPa (phi_NL = {phi_NL_iter:.2f})")

Non-Linear Creep Comparison (M_Ed = 280 kN·m)
Quasi-permanent stress limit: k_2·f_ck = 13.5 MPa

Parameter                           | Linear phi | phi_NL (1x) | phi_NL (iter)
-------------------------------------------------------------------------------
Creep coefficient used              |       2.00 |       2.12 |       2.10
Peak concrete stress (MPa)          |       14.7 |       14.5 |       14.5
Non-linear creep applied?           |         No |        Yes |        Yes
Steel stress (MPa)                  |      403.7 |      404.3 |      404.2
Crack width w_k (mm)                |      0.481 |      0.480 |      0.480

Effective Modulus Comparison:
  E_cm,eff (linear):       10946 MPa (phi = 2.00)
  E_cm,eff (phi_NL 1x):     10519 MPa (phi_NL = 2.12)
  E_cm,eff (phi_NL iter):   10586 MPa (phi_NL = 2.10)


---
## 10. Ignore Compression Steel Option

All three code checks now support the `ignore_compression_steel` parameter, which is a conservative assumption used by some commercial software. When enabled:

- **BendingCheck**: Compression steel contributes zero force to the M-N interaction diagram
- **ShearCheck**: Compression steel ignored in lever arm and rho_l calculations  
- **CrackingCheck**: Compression steel ignored in strain state solution

This provides more conservative results and can match output from software that doesn't account for compression reinforcement.

In [62]:
# Demonstrate ignore_compression_steel across all three checks
M_Ed_test = 280  # kN·m
V_Ed_test = 200  # kN
N_Ed_test = 100  # kN
M_Ed_sls = 180   # kN·m (SLS moment)

print("Comparison: With vs Without Compression Steel")
print("=" * 70)
print(f"Applied loads: M_Ed = {M_Ed_test} kN·m, V_Ed = {V_Ed_test} kN, N_Ed = {N_Ed_test} kN")
print(f"SLS moment: M_Ed = {M_Ed_sls} kN·m")
print()

# 1. Bending Check Comparison
print("1. BENDING CHECK (EC2 §6.1)")
print("-" * 40)

result_bend_with = bending_check.perform_check(M_Ed=M_Ed_test, N_Ed=N_Ed_test, ignore_compression_steel=False)
result_bend_without = bending_check.perform_check(M_Ed=M_Ed_test, N_Ed=N_Ed_test, ignore_compression_steel=True)

print(f"  {'Parameter':<25} | {'With comp. steel':>15} | {'Without':>15}")
print(f"  {'-'*25}-+-{'-'*15}-+-{'-'*15}")
print(f"  {'M_Rd (kN·m)':<25} | {result_bend_with.details['M_Rd']:>15.1f} | {result_bend_without.details['M_Rd']:>15.1f}")
print(f"  {'Utilization':<25} |  {result_bend_with.utilization:>14.1%} |  {result_bend_without.utilization:>14.1%}")
print(f"  {'Status':<25} | {result_bend_with.status.value:>15} | {result_bend_without.status.value:>15}")

# 2. Shear Check Comparison
print("\n2. SHEAR CHECK (EC2 §6.2)")
print("-" * 40)

load_case = LoadCase(V_Ed=V_Ed_test, M_Ed=M_Ed_test, N_Ed=N_Ed_test)
result_shear_with = shear_check.perform_check(load_case=load_case, ignore_compression_steel=False)
result_shear_without = shear_check.perform_check(load_case=load_case, ignore_compression_steel=True)

print(f"  {'Parameter':<25} | {'With comp. steel':>15} | {'Without':>15}")
print(f"  {'-'*25}-+-{'-'*15}-+-{'-'*15}")
print(f"  {'V_Rd (kN)':<25} | {result_shear_with.details['V_Rd']:>15.1f} | {result_shear_without.details['V_Rd']:>15.1f}")
print(f"  {'Lever arm z (mm)':<25} | {result_shear_with.details['z']:>15.1f} | {result_shear_without.details['z']:>15.1f}")
print(f"  {'Utilization':<25} |  {result_shear_with.utilization:>14.1%} |  {result_shear_without.utilization:>14.1%}")

# 3. Cracking Check Comparison
print("\n3. CRACKING CHECK (EC2 §7.3)")
print("-" * 40)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    result_crack_with = cracking_check.perform_check(M_Ed=M_Ed_sls, N_Ed=0, ignore_compression_steel=False)
    result_crack_without = cracking_check.perform_check(M_Ed=M_Ed_sls, N_Ed=0, ignore_compression_steel=True)

print(f"  {'Parameter':<25} | {'With comp. steel':>15} | {'Without':>15}")
print(f"  {'-'*25}-+-{'-'*17}-+-{'-'*17}")
print(f"  {'w_k (mm)':<25} | {result_crack_with.details.get('w_k', 0):>15.3f} | {result_crack_without.details.get('w_k', 0):>15.3f}")
print(f"  {'σ_s (MPa)':<25} | {result_crack_with.details.get('sigma_s', 0):>15.1f} | {result_crack_without.details.get('sigma_s', 0):>15.1f}")
print(f"  {'Utilization':<25} |  {result_crack_with.utilization:>14.1%} |  {result_crack_without.utilization:>14.1%}")

print("\n💡 Note: Ignoring compression steel is conservative - it reduces capacity")
print("   and increases crack widths. Use this to match conservative software.")

Comparison: With vs Without Compression Steel
Applied loads: M_Ed = 280 kN·m, V_Ed = 200 kN, N_Ed = 100 kN
SLS moment: M_Ed = 180 kN·m

1. BENDING CHECK (EC2 §6.1)
----------------------------------------
  Parameter                 | With comp. steel |         Without
  --------------------------+-----------------+----------------
  M_Rd (kN·m)               |           349.5 |           336.1
  Utilization               |           80.1% |           83.3%
  Status                    |            pass |            pass

2. SHEAR CHECK (EC2 §6.2)
----------------------------------------
  Parameter                 | With comp. steel |         Without
  --------------------------+-----------------+----------------
  V_Rd (kN)                 |           402.0 |           396.1
  Lever arm z (mm)          |           470.9 |           464.0
  Utilization               |           49.7% |           50.5%

3. CRACKING CHECK (EC2 §7.3)
----------------------------------------
  Parameter   

---
## Summary

This notebook demonstrated:

1. **BendingCheck** - First-principles M-N interaction diagram for flexural design
   - `perform_check(M_Ed, N_Ed)` for utilization
   - `plot_mn()` for interactive diagrams
   - `plot_stress_strain()` for strain/stress visualization
   - `get_moment_capacity()` for design
   - `ignore_compression_steel` option for conservative analysis

2. **ShearCheck** - Variable strut inclination method (EC2 §6.2)
   - V_Rd,c (concrete), V_Rd,s (reinforcement), V_Rd,max (strut)
   - Rigorous vs approximate lever arm modes
   - `get_required_shear_reinforcement()` for design
   - `ignore_compression_steel` option for conservative analysis

3. **CrackingCheck** - Crack width calculation (EC2 §7.3)
   - Uses characteristic (SLS) material properties
   - Accounts for tension stiffening
   - Short-term vs long-term loading effects
   - Minimum reinforcement for crack control
   - `ignore_compression_steel` option for conservative analysis

### New Features (v1.x):

4. **Linear Elastic Constitutive Model** (§7)
   - `ConcreteModelType.LINEAR_ELASTIC` for SLS analysis
   - Configurable `elastic_modulus` (defaults to `E_cm_eff` with creep)
   - Validation prevents use in ULS checks

5. **Stress Limitation Checks** (§8, EC2 §7.2)
   - Boolean flags: `check_k1_stress`, `check_k2_stress`, `check_k3_stress`, `check_yielding`, `check_k4_stress`
   - Standalone `StressLimitsCheck` class for independent stress verification

6. **Non-Linear Creep Adjustment** (§9, EC2 §3.1.4(4))
   - `apply_nonlinear_creep=True` - Auto-adjust phi when sigma_c > k_2·f_ck
   - `iterate_nonlinear_creep=True` - Iterate to convergence
   - Formula: phi_NL = phi * exp(1.5 * (k_sigma - 0.45))

7. **Ignore Compression Steel** (§10)
   - Available in all three checks via `ignore_compression_steel=True`
   - Conservative assumption matching some commercial software
   - Reduces capacity, increases crack widths